要运行此程序，请在**免费** Tesla T4 Google Colab 实例上按“*运行时*”并按“*全部运行*”！
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> 如果您需要帮助，请加入 Discord + ⭐ <i>在 <a href="https://github.com/unslothai/unsloth">Github</a> 上为我们加注星标</a> </i> ⭐
</div>

要在本地设备上安装 Unsloth，请按照 [our guide](https://unsloth.ai/docs/get-started/install) 操作。此笔记本已获得许可 [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)。

您将学习如何执行 [data prep](#Data)、如何执行 [train](#Train)、如何执行 [run the model](#Inference) 以及如何保存它

### 消息

隆重推出 **Unsloth Studio** - 一个新的开源、无代码 Web UI，用于训练和运行法学硕士。 [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<表><tr>
<tdalign="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~% 2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fupload s%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia% 26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&宽度=376&dpr=3&质量=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>训练模型</b> - 无需代码</sub></td>
<tdalign="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Z q%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26toke n%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&宽度=376&dpr=3&质量=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio 聊天 UI"></a><br><sub><b>在 Mac、Windows 和 Linux 上运行 GGUF 模型</b></sub></td>
</tr></表>

训练 MoE - DeepSeek、GLM、Qwen 和 gpt-oss 速度提高 12 倍，VRAM 减少 35%。 [Blog](https://unsloth.ai/docs/new/faster-moe)

超长上下文强化学习现已推出，上下文窗口增加了 7 倍！ [Blog](https://unsloth.ai/docs/new/grpo-long-context)

强化学习新功能：[FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

请访问我们的文档以获取所有 [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) 和 [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks)。

### 安装

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.7.1" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.5"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo

#  Mamba is supported only on torch==2.7.1. If you have newer torch versions, please wait 30 minutes!
!uv pip install --no-build-isolation mamba_ssm==2.2.5 causal_conv1d==1.5.2
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

### 不懒惰

In [ ]:
from unsloth import FastLanguageModel
import torch

fourbit_models = [
    "unsloth/granite-4.0-micro",
    "unsloth/granite-4.0-h-micro",
    "unsloth/granite-4.0-h-tiny",
    "unsloth/granite-4.0-h-small",

    # 基础预训练 Granite 4 模型
    "unsloth/granite-4.0-micro-base",
    "unsloth/granite-4.0-h-micro-base",
    "unsloth/granite-4.0-h-tiny-base",
    "unsloth/granite-4.0-h-small-base",

    # 4 位动态量化可实现卓越的精度和低内存使用量
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/Phi-4",
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/orpheus-3b-0.1-ft-unsloth-bnb-4bit" # [新]我们支持 TTS 模型！
] # 更多模型请访问 https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/granite-4.0-h-micro",
    max_seq_length = 2048,   # 选择任何一个用于长上下文！
    load_in_4bit = False,    # 4 位量化以减少内存
    load_in_8bit = False,    # [新！] 更准确一点，使用 2 倍内存
    full_finetuning = False, # [新！] 我们现在进行了全面的微调！
)

==((====))==  Unsloth 2025.9.11: Fast Granitemoehybrid patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


The fast path for GraniteMoeHybrid will be used when running the model on a GPU


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

我们现在添加了 LoRA 适配器，因此我们只需要更新少量参数！

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # 选择任意大于 0 的数字！建议 8、16、32、64、128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",
                      "shared_mlp.input_linear", "shared_mlp.output_linear"],
    lora_alpha = 32,
    lora_dropout = 0, # 支持任意，但 = 0 已优化
    bias = "none",    # 支持任何，但=“无”已优化
    # [新]“unsloth”使用的 VRAM 减少了 30%，适合 2 倍大的批量大小！
    use_gradient_checkpointing = "unsloth", # 对于很长的上下文来说是真实的或“不懒惰的”
    random_state = 3407,
    use_rslora = False,  # 我们支持排名稳定的 LoRA
    loftq_config = None, # 和阁楼Q
)

Unsloth: Making `model.base_model.model.model` require gradients


<a名称=“数据”></a>
### 数据准备
#### 📄 使用 Google Sheets 作为训练数据
我们的目标是创建一个能够主动帮助和解决问题的客户支持机器人。

我们将示例存储在包含两列的 Google 表格中：

- **片段**：简短的客户支持互动
- **建议**：关于代理应如何回应的建议

这使事情变得简单和协作。任何人都可以编辑该工作表，无需设置数据库。  
<br>

---
<br>

#### 🔍 为什么采用这种格式？

此设置非常适合以下任务：

- `输入片段 → 建议回复`
- `提示→重写`
- `错误报告 → 诊断`
- `文本 → 标签或类别`

只需在电子表格中收集示例，您就可以获得可用的训练数据。  
<br>

---
<br>

#### ✅ 你将学到什么

我们将展示如何：

1. 将 Google Sheet 加载到笔记本中
2.将其格式化为数据集
3. 用它来培训或提示法学硕士


Granite-4 的聊天模板如下所示：
```
<|start_of_role|>system<|end_of_role|>Knowledge Cutoff Date: April 2024.
Today's Date: June 24, 2025.
You are Granite, developed by IBM. You are a helpful AI assistant.<|end_of_text|>

<|start_of_role|>user<|end_of_role|>How do astronomers determine the original wavelength of light emitted by a celestial body at rest, which is necessary for measuring its speed using the Doppler effect?<|end_of_text|>

<|start_of_role|>assistant<|end_of_role|>Astronomers make use of the unique spectral fingerprints of elements found in stars...<|end_of_text|>
```

In [ ]:
from datasets import load_dataset, Dataset

# 使用下面的共享表
# sheet_url = 'https://docs.google.com/spreadsheets/d/1NrjI5AGNIwRtKTAse5TW_hWq2CwAS03qCHif6vaaRh0/export?format=csv&gid=0'

# 或者不懒惰/支持机器人推荐
sheet_url = "https://huggingface.co/datasets/unsloth/Support-Bot-Recommendation/raw/main/support_recs.csv"

dataset = load_dataset(
    "csv",
    data_files = {"train": sheet_url},
    column_names = ["snippet", "recommendation"], # 替换为工作表的实际列名称
    skiprows = 1  # 跳过标题行
)["train"]

Generating train split: 0 examples [00:00, ? examples/s]

我们刚刚将 Google 表格加载为 csv 样式数据集，但我们仍然需要将其格式化为如下所示的对话样式，然后应用聊天模板。

```
{"role": "system", "content": "You are an assistant"}
{"role": "user", "content": "What is 2+2?"}
{"role": "assistant", "content": "It's 4."}
```

我们将使用辅助函数“formatting_prompts_func”来完成这两件事！

In [ ]:
def formatting_prompts_func(examples):
    user_texts = examples['snippet']
    response_texts = examples['recommendation']
    messages = [
        [{"role": "user", "content": user_text},
        {"role": "assistant", "content": response_text}] for user_text, response_text in zip(user_texts, response_texts)
    ]
    texts = [tokenizer.apply_chat_template(message, tokenize = False, add_generation_prompt = False) for message in messages]

    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True,)

Map:   0%|          | 0/504 [00:00<?, ? examples/s]

我们现在在格式化之前查看原始输入数据。

In [ ]:
dataset[5]["snippet"]

'User: I\'m getting an error when trying to log in. \nAgent: What error message are you seeing? \nUser: It says "Invalid credentials" even though I\'m sure my password is correct. \nAgent: Have you tried clearing your browser cache? \nUser: Yes, I cleared it already. \nAgent: Let me check your account status. \nUser: I\'ve been using this account for months without issues. \nAgent: I found no issues with your account. \nUser: Maybe there\'s a problem with the login server? \nAgent: Let\'s try resetting your password. \nUser: I just did that, and it\'s not working either. \nAgent: I\'ll need to escalate this to our engineering team. \nUser: Okay, what should I do in the meantime? \nAgent: Try using a different browser or device. \nUser: I\'ll try Chrome on my laptop. \nAgent: Let me know if that resolves the issue.'

In [ ]:
dataset[5]['recommendation']

'#### Analysis\nThe user is experiencing persistent login issues ("Invalid credentials", password reset failure) despite clearing cache and confirming correct credentials. No account anomalies were detected by the agent. The root cause remains unresolved and potentially related to server-side authentication or user-specific credential handling.\n\n#### Recommendation\n- Step 1: Confirm if using Chrome on the laptop resolved the login issue. (User action)\n- Step 2: If Step 1 was successful, no further immediate action needed. If not, proceed to escalate based on user feedback.\n- *Next best action for the agent*: Report back to the user whether using Chrome on their laptop confirmed or failed to resolve the issue.'

我们看到聊天模板如何改变这些对话。

In [ ]:
dataset[5]["text"]

'<|start_of_role|>user<|end_of_role|>User: I\'m getting an error when trying to log in. \nAgent: What error message are you seeing? \nUser: It says "Invalid credentials" even though I\'m sure my password is correct. \nAgent: Have you tried clearing your browser cache? \nUser: Yes, I cleared it already. \nAgent: Let me check your account status. \nUser: I\'ve been using this account for months without issues. \nAgent: I found no issues with your account. \nUser: Maybe there\'s a problem with the login server? \nAgent: Let\'s try resetting your password. \nUser: I just did that, and it\'s not working either. \nAgent: I\'ll need to escalate this to our engineering team. \nUser: Okay, what should I do in the meantime? \nAgent: Try using a different browser or device. \nUser: I\'ll try Chrome on my laptop. \nAgent: Let me know if that resolves the issue.<|end_of_text|>\n<|start_of_role|>assistant<|end_of_role|>#### Analysis\nThe user is experiencing persistent login issues ("Invalid credent

<a name="火车"></a>
### 训练模型
现在让我们训练我们的模型。我们执行 60 个步骤来加快速度，但您可以设置“num_train_epochs=1”以进行完整运行，并关闭“max_steps=None”。

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # 可以设置评价！
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # 使用 GA 来模拟批量大小！
        warmup_steps = 5,
        # num_train_epochs = 1, # 将其设置为 1 次完整训练运行。
        max_steps = 60,
        learning_rate = 2e-4, # 长时间训练时减少至 2e-5
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # 使用 TrackIO/WandB 等
    ),
)

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/504 [00:00<?, ? examples/s]

我们还使用 Unsloth 的“train_on_completions”方法仅对辅助输出进行训练，并忽略用户输入的损失。这有助于提高微调的准确性！

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_of_role|>user<|end_of_role|>",
    response_part = "<|start_of_role|>assistant<|end_of_role|>",
)

Map (num_proc=2):   0%|          | 0/504 [00:00<?, ? examples/s]

让我们验证一下指令部分的屏蔽是否已完成！让我们再次打印第 100 行。

In [ ]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

'<|start_of_role|>user<|end_of_role|>User: My account is locked. I tried to log in but got an error message saying "Too many failed attempts".\n\nAgent: Can you please try logging in again and enter the security code sent to your email? That should unlock your account temporarily.\n\nUser: I did that already. I received the code and entered it, but my account is still locked.\n\nAgent: I see. Have you tried resetting your password via the \'Forgot Password\' link?\n\nUser: Yes, I clicked on that. It sent me an email with a reset link, but when I tried to reset my password, I got an error message saying "Invalid request".\n\nAgent: Okay, I can\'t access your account to check directly. Could you please provide me with your account ID or email address associated with the account?\n\nUser: My email is user@example.com. Account ID is 123456789.\n\nAgent: Thank you. I\'m looking into this. It seems there might be an issue with the account lockout mechanism or the password reset process. I\'l

现在让我们打印屏蔽的示例 - 您应该只看到存在的答案：

In [ ]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

"                                                                                                                                                                                                                                                 #### Analysis\nThe user is experiencing account lockout due to multiple failed login attempts, and standard troubleshooting steps like password reset and security code entry are failing, indicating a potential issue with the account lockout mechanism or password recovery system.\n\n#### Recommendation\n- Step 1: Attempt to reset the password using the 'Forgot Password' link and provide the error details received.\n- Step 2: Contact support with the account ID/email and request manual account unlock and investigation.\n- *Next best action for the agent*: Instruct the user to contact support immediately, providing their account details for manual intervention and further investigation.<|end_of_text|>\n"

In [ ]:
# @title 显示当前内存统计信息
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
6.059 GB of memory reserved.


让我们训练模型吧！要恢复训练运行，请设置“trainer.train(resume_from_checkpoint = True)”

```
Notice you might have to wait ~10 minutes for the Mamba kernels to compile! Please be patient!
```

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 504 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 1,703,936 of 3,193,100,032 (0.05% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,0.893200
2,1.193700
3,1.385300
4,1.949300
5,1.909700
6,2.066900
7,1.923100
8,2.053300
9,1.503300
10,1.325700


In [ ]:
# @title 显示最终内存和时间统计数据
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

954.0727 seconds used for training.
15.9 minutes used for training.
Peak reserved memory = 10.42 GB.
Peak reserved memory for training = 4.361 GB.
Peak reserved memory % of max memory = 70.687 %.
Peak reserved memory for training % of max memory = 29.584 %.


<a name="推理"></a>
### 推论
让我们通过 Unsloth 原生推理来运行模型！我们将使用训练数据中未包含的一些示例片段来了解所学内容。

In [ ]:
# @title 测试场景
# ---场景1：视频会议屏幕共享Bug（11回合）---
scenario_1 = """
User: Everyone in my meeting just sees a black screen when I share.
Agent: Sorry about that—are you sharing a window or your entire screen?
User: Entire screen on macOS Sonoma.
Agent: Thanks. Do you have “Enable hardware acceleration” toggled on in Settings → Video?
User: Yeah, that switch is on.
Agent: Could you try toggling it off and start a quick test share?
User: Did that—still black for attendees.
Agent: Understood. Are you on the desktop app v5.4.2 or the browser client?
User: Desktop v5.4.2—just updated this morning.
"""

# --- 场景 2：智能锁低电量循环（9 圈） ---
scenario_2 = """
User: I changed the batteries, but the lock app still says 5 % and won’t auto-lock.
Agent: Let’s check firmware. In the app, go to Settings → Device Info—what version shows?
User: 3.18.0-alpha.
Agent: Latest stable is 3.17.5. Did you enroll in the beta program?
User: I might have months ago.
Agent: Beta builds sometimes misreport battery. Remove one battery, wait ten seconds, reinsert, and watch the LED pattern.
User: LED blinks blue twice, then red once.
Agent: That blink code means “config mismatch.” Do you still have the old batteries handy?
User: Tossed them already.
"""

# --- 场景 3：会计 SaaS — 损坏的发票导出（10 次） ---
scenario_3 = """
User: Every invoice I download today opens as a blank PDF.
Agent: Is this happening to historic invoices, new ones, or both?
User: Both. Anything I export is 0 bytes.
Agent: Are you exporting through “Bulk Actions” or individual invoice pages?
User: Individual pages.
Agent: Which browser/OS combo?
User: Chrome on Windows 11, latest update.
Agent: We released a new PDF renderer at 10 a.m. UTC. Could you try Edge quickly, just to rule out a caching quirk?
User: Tried Edge—same zero-byte file.
"""

# --- 场景 4：健身追踪器应用程序 — 计步卡住（8 圈）---
scenario_4 = """
User: My step count has been frozen at 4,237 since last night.
Agent: Which phone are you syncing with?
User: iPhone 15, iOS 17.5.
Agent: In the Health Permissions screen, does “Motion & Fitness” show as ON?
User: Yes, it’s toggled on.
Agent: When you pull down to refresh the dashboard, does the sync spinner appear?
User: Spinner flashes for a second, then nothing changes.
"""

# --- 场景 5：在线课程平台 — 测验提交错误（12 轮） ---
scenario_5 = """
User: My quiz submits but then shows “Unknown grading error” and resets the answers.
Agent: Which course and quiz name?
User: History 301, Unit 2 Quiz.
Agent: Do you notice a red banner or any code like GR-# ## 在角落里？
User: Banner says “GR-412”.
Agent: That code points to answer-payload size. Were you pasting images or long text into any answers?
User: Maybe a long essay—about 800 words in Question 5.
Agent: Are you on a laptop or mobile?
User: Laptop, Safari on macOS.
"""

In [ ]:
FastLanguageModel.for_inference(model) # 使本机推理速度提高 2 倍

messages = [
    {"role": "user", "content": scenario_1},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # 必须添加生成
    padding = True,
    return_tensors = "pt",
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = False)

_ = model.generate(**inputs,
                   streamer = text_streamer,
                   max_new_tokens = 512, # 如果代币被切断则增加
                   use_cache = True,
                   # 根据您的喜好调整采样参数
                   do_sample = True,
                   temperature = 0.7, top_p = 0.8, top_k = 20,
)

<|start_of_role|>user<|end_of_role|>
User: Everyone in my meeting just sees a black screen when I share.
Agent: Sorry about that—are you sharing a window or your entire screen?
User: Entire screen on macOS Sonoma.
Agent: Thanks. Do you have “Enable hardware acceleration” toggled on in Settings → Video?
User: Yeah, that switch is on.
Agent: Could you try toggling it off and start a quick test share?
User: Did that—still black for attendees.
Agent: Understood. Are you on the desktop app v5.4.2 or the browser client?
User: Desktop v5.4.2—just updated this morning.
<|end_of_text|>
<|start_of_role|>assistant<|end_of_role|>#### Analysis
- The user reports a black screen for attendees when sharing the entire screen on macOS Sonoma.
- The user has "Enable hardware acceleration" enabled in Settings → Video.
- The user is on the desktop app v5.4.2, which was updated this morning.

#### Recommendation
- Step 1: Ask the user to restart the desktop app and try sharing again.
- Step 2: If the issue 

In [ ]:
FastLanguageModel.for_inference(model) # 使本机推理速度提高 2 倍

messages = [
    {"role": "user", "content": scenario_2},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # 必须添加生成
    padding = True,
    return_tensors = "pt",
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = False)

_ = model.generate(**inputs,
                   streamer = text_streamer,
                   max_new_tokens = 512, # 如果代币被切断则增加
                   use_cache = True,
                   # 根据您的喜好调整采样参数
                   do_sample = False,
                   temperature = 0.7, top_p = 0.8, top_k = 20,
)

<|start_of_role|>user<|end_of_role|>
User: I changed the batteries, but the lock app still says 5 % and won’t auto-lock.
Agent: Let’s check firmware. In the app, go to Settings → Device Info—what version shows?
User: 3.18.0-alpha.
Agent: Latest stable is 3.17.5. Did you enroll in the beta program?
User: I might have months ago.
Agent: Beta builds sometimes misreport battery. Remove one battery, wait ten seconds, reinsert, and watch the LED pattern.
User: LED blinks blue twice, then red once.
Agent: That blink code means “config mismatch.” Do you still have the old batteries handy?
User: Tossed them already.
<|end_of_text|>
<|start_of_role|>assistant<|end_of_role|>#### Analysis
The user's lock app shows 5% battery and fails to auto-lock, despite recent battery replacement. The firmware version is 3.18.0-alpha, which is outdated. The LED blink pattern indicates a configuration mismatch, likely due to the user's previous beta enrollment.

#### Recommendation
1. **Verify firmware version**

<a name="保存"></a>
### 保存、加载微调模型
要将最终模型保存为 LoRA 适配器，请使用 Hugging Face 的 `push_to_hub` 进行在线保存，或使用 `save_pretrained` 进行本地保存。

**[注意]** 这仅保存 LoRA 适配器，而不是完整模型。要保存到 16 位或 GGUF，请向下滚动！

In [ ]:
model.save_pretrained("granite_lora")  # 本地储蓄
tokenizer.save_pretrained("granite_lora")
# model.push_to_hub("your_name/granite_lora", token = "YOUR_HF_TOKEN") # 在线保存
# tokenizer.push_to_hub("your_name/granite_lora", token = "YOUR_HF_TOKEN") # 在线保存

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/chat_template.jinja',
 'lora_model/vocab.json',
 'lora_model/merges.txt',
 'lora_model/added_tokens.json',
 'lora_model/tokenizer.json')

现在，如果您想加载我们刚刚保存用于推理的 LoRA 适配器，请将“False”设置为“True”：

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "granite_lora", # 您用于训练的模型
        max_seq_length = 2048,
        load_in_4bit = True,
    )

### 保存为 VLLM 的 float16

我们还支持直接保存到`float16`。对于 float16 选择“merged_16bit”，对于 int4 选择“merged_4bit”。我们还允许“lora”适配器作为后备。使用 `push_to_hub_merged` 上传到您的 Hugging Face 帐户！您可以前往 https://huggingface.co/settings/tokens 获取您的个人代币。有关更多部署选项，请参阅 [our docs](https://unsloth.ai/docs/basics/inference-and-deployment)。

In [ ]:
# 合并到16位
if False:
    model.save_pretrained_merged("granite_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: # 推送至 HF 集线器
    model.push_to_hub_merged("HF_USERNAME/granite_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# 合并到4bit
if False:
    model.save_pretrained_merged("granite_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: # 推送至 HF 集线器
    model.push_to_hub_merged("HF_USERNAME/granite_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# 只需 LoRA 适配器
if False:
    model.save_pretrained("granite_lora")
    tokenizer.save_pretrained("granite_lora")
if False: # 推送至 HF 集线器
    model.push_to_hub("HF_USERNAME/granite_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/granite_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp 转换
要保存到 `GGUF` / `llama.cpp`，我们现在原生支持它！我们克隆 `llama.cpp` 并默认将其保存到 `q8_0`。我们允许像“q4_k_m”这样的所有方法。使用`save_pretrained_gguf`进行本地保存，使用`push_to_hub_gguf`上传到HF。

一些支持的量化方法（完整列表在我们的 [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf) 上）：
* `q8_0` - 快速转换。资源使用率较高，但总体上可以接受。
* `q4_k_m` - 推荐。使用Q6_K作为attention.wv和feed_forward.w2张量的一半，否则使用Q4_K。
* `q5_k_m` - 推荐。使用Q6_K作为attention.wv和feed_forward.w2张量的一半，否则使用Q5_K。

[**新**] 要微调并自动导出到 Ollama，请尝试我们的 [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

同样，如果您想将 GGUF 推送到您的 Hugging Face 帐户，请将“if False”设置为“if True”并添加您的 Hugging Face 令牌和上传位置！

In [ ]:
# 保存到8位Q8_0
if False:
    model.save_pretrained_gguf("granite_finetune", tokenizer,)
# 记得去 https://huggingface.co/settings/tokens 获取令牌！
# 并将 hf 更改为您的用户名！
if False:
    model.push_to_hub_gguf("HF_USERNAME/granite_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# 保存到16位GGUF
if False:
    model.save_pretrained_gguf("granite_finetune", tokenizer, quantization_method = "f16")
if False: # 推送至 HF 集线器
    model.push_to_hub_gguf("HF_USERNAME/granite_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# 保存到 q4_k_m GGUF
if False:
    model.save_pretrained_gguf("granite_finetune", tokenizer, quantization_method = "q4_k_m")
if False: # 推送至 HF 集线器
    model.push_to_hub_gguf("HF_USERNAME/granite_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# 保存到多个 GGUF 选项 - 如果您想要多个，速度会更快！
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/granite_finetune", # 将 hf 更改为您的用户名！
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN", # 在 https://huggingface.co/settings/tokens 获取令牌
    )

现在，使用 llama.cpp 中的 `granite_finetune.Q8_0.gguf` 文件或 `granite_finetune.Q4_K_M.gguf` 文件。

我们就完成了！如果您对 Unsloth 有任何疑问，我们有 [Discord](https://discord.gg/unsloth) 频道！如果您发现任何错误或想要随时更新最新的 LLM 内容，或需要帮助、加入项目等，请随时加入我们的 Discord！

其他一些资源：
1.训练自己的推理模型——Llama GRPO笔记本[Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. 将微调保存到Ollama。 [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 视觉微调 - 射线照相用例。 [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. 请参阅我们的 [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks) 上的 DPO、ORPO、持续预训练、对话微调等笔记本！

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  如果您需要帮助，请加入 Discord + ⭐️ <i>在 <a href="https://github.com/unslothai/unsloth">Github</a> 上为我们加注星标</i> ⭐️
</div>

  此笔记本和所有 Unsloth 笔记本均已获得 [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme) 许可。